# RNNs with PyTorch — Character-Level Language Model

An RNN processes a sequence one step at a time, maintaining a **hidden state** that summarises past context: `h_t = f(x_t, h_{t-1})`.

**Corpus**: *Alice's Adventures in Wonderland* (Lewis Carroll, 1865) via NLTK Gutenberg.

We train a character-level model: given the last 100 characters, predict the next one. After training we can generate new text by feeding predictions back as input.

## Step 1: Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from nltk.corpus import gutenberg

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

## Step 2: Load and Inspect the Corpus

We take the first 80 000 characters of *Alice* — enough text to learn real English structure while keeping training fast.

In [ ]:
text = gutenberg.raw('carroll-alice.txt')[:80000]

# Build character vocabulary
char2idx = {ch: i for i, ch in enumerate(sorted(set(text)))}
idx2char  = {i: ch for ch, i in char2idx.items()}
vocab_size = len(char2idx)

print(f'Characters : {len(text):,}')
print(f'Vocab size : {vocab_size} unique chars')
print(f'Sample     : {text[:120]!r}')

## Step 3: Create Training Sequences

Slide a window of length `SEQ_LEN=100` across the text with step `STRIDE=5`, producing *(input, target)* pairs where target is the input shifted by one position.

In [ ]:
SEQ_LEN = 100
STRIDE  = 5

data = [char2idx[c] for c in text]

# TODO: build X_seqs and y_seqs lists of lists
# X_seqs[i] = data[i*STRIDE : i*STRIDE + SEQ_LEN]
# y_seqs[i] = data[i*STRIDE+1 : i*STRIDE + SEQ_LEN + 1]
X_seqs = ...
y_seqs = ...

X_tensor = torch.tensor(X_seqs, dtype=torch.long)
y_tensor = torch.tensor(y_seqs, dtype=torch.long)
print(f'Sequences: {len(X_seqs):,}  X:{X_tensor.shape}  y:{y_tensor.shape}')

## Step 4: DataLoader

Batch the sequences. We shuffle here because the model will still see the correct temporal ordering *within* each sequence.

In [ ]:
# TODO: create TensorDataset + DataLoader with batch_size=128, shuffle=True
train_loader = ...
print(f'Batches per epoch: {len(train_loader)}')

## Step 5: Define the RNN Model

Instead of one-hot encoding, we use `nn.Embedding` to map each character index to a learned dense vector. This is more compact and gives the model more expressive power.

`nn.RNN(embed_dim, hidden_size, num_layers, batch_first=True)` returns `(output, h_n)` where `output` has shape `(batch, seq_len, hidden_size)`.

In [ ]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_size=256, num_layers=2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        # TODO: embedding, rnn (with dropout=0.3), fc
        self.embedding = ...
        self.rnn       = ...
        self.fc        = ...

    def forward(self, x, hidden=None):
        x = self.embedding(x)            # (B, L, E)
        out, hidden = self.rnn(x, hidden) # out: (B, L, H)
        # TODO: pass through fc to get logits (B, L, vocab_size)
        logits = ...
        return logits, hidden

    def init_hidden(self, batch_size, device):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size, device=device)

model = CharRNN(vocab_size).to(device)
print(model)
print('Parameters:', sum(p.numel() for p in model.parameters()))

## Step 6: Training Loop

Key details:
- Reshape logits to `(B*L, vocab_size)` and targets to `(B*L,)` for `CrossEntropyLoss`
- **Gradient clipping** (`clip_grad_norm_`) prevents exploding gradients — a common   problem with RNNs on long sequences

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)

for epoch in range(30):
    model.train()
    total_loss = 0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        # TODO: forward pass
        logits, _ = ...
        # TODO: reshape and compute loss
        loss = ...
        optimizer.zero_grad()
        loss.backward()
        # TODO: clip gradients (max_norm=1.0)
        ...
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d} | Loss: {total_loss/len(train_loader):.4f}')

## Step 7: Generate Text

Feed a seed string to warm up the hidden state, then sample character by character. The `temperature` parameter controls randomness: lower = more conservative, higher = more creative.

In [ ]:
def generate(model, seed_text, char2idx, idx2char, length=400,
             temperature=0.8, device='cpu'):
    model.eval()
    with torch.no_grad():
        hidden = model.init_hidden(1, device)
        # Warm up with seed (all chars except last)
        for ch in seed_text[:-1]:
            # TODO: forward pass for each seed char
            ...
        result = seed_text
        current = seed_text[-1]
        for _ in range(length):
            # TODO: encode current char, forward, sample next char
            ...
    return result

print(generate(model, 'Alice ', char2idx, idx2char, length=400,
               temperature=0.8, device=str(device)))

## Step 8: Experiment with Temperature

In [ ]:
# Low temperature — more predictable / repetitive
print('=== temperature=0.5 ===')
print(generate(model, 'The ', char2idx, idx2char, 200, temperature=0.5, device=str(device)))

# High temperature — more random / creative
print('\n=== temperature=1.2 ===')
print(generate(model, 'The ', char2idx, idx2char, 200, temperature=1.2, device=str(device)))